# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [1]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


Recordar que tengo que hacer una copia


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [28]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [29]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama","serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
  try:
    requests.get("http://localhost:11434")
    print("Servidor de Ollama listo.")
    break
  except requests.exceptions.ConnectionError:
      time.sleep(1)
else:
        print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.6


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [14]:
 # Descargar los dos modelos Llama (la primera vez tarda unos minutos)
MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    1 second ago              


### **CLIENTE DE PYTHON**

In [17]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [18]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento de una computadora?"

response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento de una computadora?
**Diferencia entre la RAM y el almacenamiento de una computadora**

La RAM (Random Access Memory) y el almacenamiento de una computadora son dos conceptos importantes que se relacionan estrechamente con el rendimiento y la capacidad de procesar información.

**RAM (Random Access Memory)**

La RAM es un tipo de memoria temporal que se utiliza para almacenar datos y programas en la memoria principal de una computadora. Su función es proporcionar acceso rápido y directo a la información necesaria para procesar, lo que permite que la computadora realice tareas con mayor eficiencia.

La RAM es volátil, lo que significa que se pierde su contenido cuando la computadora se apaga. Sin embargo, cuando se enciende de nuevo, la computadora carga la RAM con el contenido de la última sesión.

**Almacenamiento de una computadora**

El almacenamiento de una computadora, por otro lado, se refiere a cualquier tipo de dispositivo

In [19]:
print(response.model_dump_json(indent=2))

{
  "model": "llama3.2:3b",
  "created_at": "2026-10-18T16:13:59.030927582Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 106593707735,
  "load_duration": 59686615569,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 40286096000,
  "eval_count": 541,
  "eval_duration": 6617196000,
  "message": {
    "role": "assistant",
    "content": "**Diferencia entre la RAM y el almacenamiento de una computadora**\n\nLa RAM (Random Access Memory) y el almacenamiento de una computadora son dos conceptos importantes que se relacionan estrechamente con el rendimiento y la capacidad de procesar información.\n\n**RAM (Random Access Memory)**\n\nLa RAM es un tipo de memoria temporal que se utiliza para almacenar datos y programas en la memoria principal de una computadora. Su función es proporcionar acceso rápido y directo a la información necesaria para procesar, lo que permite que la computadora realice tareas con mayor eficiencia.\n\nLa RAM es volátil, lo que significa que se pier

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [20]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)

Tokens del prompt: 43
Tokens de la respuesta: 541
Tokens totales: 584


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [21]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
inicio = time.time()
response_tiempo= client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio

print(f"Tiempo de respuesta:  {duracion:.2f} segundos")

Tiempo de respuesta:  10.54 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [30]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9 #nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s - {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s - {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)


Modelo ligero (llama3.2:3b): 10.54 s - 584 tokens
Modelo grande (llama3.1:8b): 23.85 s - 230 tokens (de los cuales 18.42 s fueron carga del modelo)

Respuesta del modelo grande:
 La RAM (Memoria de Acceso Aleatorio) es una memoria que la computadora utiliza para ejecutar sus programas. La RAM es temporal y se utiliza mientras la computadora está en marcha. A medida que la computadora sigue ejecutando un programa, el programa es reemplazado por otro. Cuando la computadora se apaga, la información en la RAM se borra. 

El almacenamiento de la computadora es un tipo de dispositivo que almacena información y datos a largo plazo. El almacenamiento puede ser en forma de discos duros, unidades de estado sólido (SSD) u otros dispositivos que almacenan datos en la computadora. Los datos almacenados en el almacenamiento se conservan incluso después de que la computadora se apague. 

En resumen, la RAM se utiliza para ejecutar programas y tareas de forma temporal, mientras que el almacenamiento s

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [31]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado para Challenge.")

Ollama instalado para Challenge.


In [32]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama","serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
  try:
    requests.get("http://localhost:11434")
    print("Servidor de Ollama listo.")
    break
  except requests.exceptions.ConnectionError:
      time.sleep(1)
else:
        print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.6


In [35]:
# Descargar el modelo ligero de llama e inicializar el cliente
MODELO_LIGERO = "llama3.2:3b"

!ollama pull {MODELO_LIGERO}
!ollama list

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente Inicializado y Modelo Ligero descargado")


NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    
llama3.1:8b    46e0c10c039e    4.9 GB    41 minutes ago            
Cliente Inicializado y Modelo Ligero descargado


**2. Consulta la primera pregunta:**

* Construye una lista llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).
*  Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [57]:
# Definir la lista de preguntas
preguntas=[]
preguntas.append("¿Qué es llama en IA?")
preguntas.append("¿Qué es Java?")
preguntas.append("¿Que es Python")

print("Preguntas guardadas en lista")


Preguntas guardadas en lista


In [58]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1
prompt = preguntas[0]
inicio = time.time()
response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
tokens_totales = response.prompt_eval_count + response.eval_count

resultado_1 = {
    "respuesta":response.message.content,
    "tiempo_respuesta":duracion,
    "tokens":tokens_totales
}
print("Respuesta de Pregunta 1 guardada en Diccionario")

Respuesta de Pregunta 1 guardada en Diccionario


**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [59]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2
prompt = preguntas[1]
inicio = time.time()
response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
tokens_totales = response.prompt_eval_count + response.eval_count

resultado_2 = {
    "respuesta":response.message.content,
    "tiempo_respuesta":duracion,
    "tokens":tokens_totales
}
print("Respuesta de Pregunta 2 guardada en Diccionario")

Respuesta de Pregunta 2 guardada en Diccionario


In [60]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3
prompt = preguntas[2]
inicio = time.time()
response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
tokens_totales = response.prompt_eval_count + response.eval_count

resultado_3 = {
    "respuesta":response.message.content,
    "tiempo_respuesta":duracion,
    "tokens":tokens_totales
}
print("Respuesta de Pregunta 3 guardada en Diccionario")

Respuesta de Pregunta 3 guardada en Diccionario


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [61]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)
print("Resultados guardados en Lista")

Resultados guardados en Lista


**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [63]:
# Mostrar la tabla final de resultados
import json
print(json.dumps(resultados, indent=2))
print("El modelo ligero ha resuelto las 3 preguntas de forma satisfactoria")

[
  {
    "respuesta": "**\u00bfQu\u00e9 es Llama en IA?**\n\nLlama es un modelo de inteligencia artificial desarrollado por Meta, que es capaz de entender y generar lenguaje humano de manera natural. Fue lanzado en noviembre de 2022 y r\u00e1pidamente se convirti\u00f3 en una de las herramientas de inteligencia artificial m\u00e1s avanzadas disponibles.\n\n**Caracter\u00edsticas clave de Llama**\n\nLlama se basa en un algoritmo de aprendizaje autom\u00e1tico que utiliza redes neuronales convolucionales y recurrentes para analizar y generar texto. Algunas de las caracter\u00edsticas clave de Llama incluyen:\n\n*   **Capacidad de entender lenguaje**: Llama puede entender el significado de las palabras y las frases, lo que le permite generar texto coherente y relevante.\n*   **Capacidad de generar texto**: Llama puede generar texto de manera natural, incluyendo historias, descripciones y di\u00e1logos.\n*   **Capacidad de aprender**: Llama puede aprender de grandes cantidades de datos y 